# Assembling CRN Metadata

### Inputs
3 v5 release tables from …/asap-curated-cohort-pmdbs-sc-rnaseq/metadata/release/v5.0.0/:
`SAMPLE, SUBJECT, CLINPATH`

In [ ]:
from pathlib import Path
from typing import Sequence
import numpy as np
import pandas as pd
HOME = Path.home()
WS_ROOT = HOME / "workspace"
CRN_METADATA_DIR = ( WS_ROOT / "ASAPCRN_PMDBS_scRNAseq" / "pmdbs-sc-rnaseq-v3"/ "cohort-pmdbs-sc-rnaseq" / "metadata")
OUTPUT_DIR = WS_ROOT / "ws_files" / "ASAPxCRN" / "metadata_assembly"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("CRN metadata:", CRN_METADATA_DIR)
print("Outputs:", OUTPUT_DIR)

## Load Metadata Tables

In [ ]:
# asap-crn
SAMPLE = pd.read_csv(CRN_METADATA_DIR / "SAMPLE.csv", low_memory=False)
SUBJECT = pd.read_csv(CRN_METADATA_DIR / "SUBJECT.csv", low_memory=False)
CLINPATH = pd.read_csv(CRN_METADATA_DIR / "CLINPATH.csv", low_memory=False)
source_tables = {
    "STUDY" : STUDY,
    "SAMPLE" : SAMPLE,
    "SUBJECT": SUBJECT,
    "CLINPATH": CLINPATH,
}
print("SAMPLE:", SAMPLE.shape)
print("SUBJECT:", SUBJECT.shape)
print("CLINPATH:", CLINPATH.shape)

## Define Functions for Merging Metadata

In [ ]:
## Functions for merging tables
MISSING_LABELS = {
    "",
    "NA",
    "N/A",
    "NAN",
    "NONE",
    "NULL",
    "NOT REPORTED",
    "UNKNOWN",
}
def standardize_missing(data: pd.DataFrame) -> pd.DataFrame:
    """Convert blank strings and common missing labels to pd.NA."""
    result = data.copy()
    text_columns = result.select_dtypes(
        include=["object", "string"]
    ).columns
    for column in text_columns:
        cleaned = result[column].astype("string").str.strip()
        missing_mask = cleaned.str.upper().isin(MISSING_LABELS)
        result[column] = cleaned.mask(missing_mask, pd.NA)
    return result
def collapse_duplicate_records(
    data: pd.DataFrame,
    keys: Sequence[str],
    table_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Collapse duplicate keys while preserving agreed complementary values.
    Selection and fill rules:
    1. Select the row with the most populated non-key fields as the base.
    2. If a field has exactly one distinct non-null value in the group,
       place that value in the collapsed row.
    3. If a field has multiple distinct values, retain the selected base
       value and record all observed values in the conflict report.
    4. Record ties when multiple rows are equally complete.
    """
    work = standardize_missing(data)
    metadata_columns = [
        column for column in work.columns
        if column not in keys
    ]
    work["_fields_present"] = (
        work[metadata_columns]
        .notna()
        .sum(axis=1)
    )
    work["_source_row"] = np.arange(len(work))
    selected_records = []
    conflict_records = []
    tie_records = []
    group_key = keys[0] if len(keys) == 1 else list(keys)
    for key_values, group in work.groupby(
        group_key,
        dropna=False,
        sort=False,
    ):
        if not isinstance(key_values, tuple):
            key_values = (key_values,)
        key_record = dict(zip(keys, key_values))
        max_fields = int(group["_fields_present"].max())
        top_rows = (
            group.loc[
                group["_fields_present"].eq(max_fields)
            ]
            .sort_values("_source_row")
        )
        selected = top_rows.iloc[0].copy()
        if len(top_rows) > 1:
            tie_records.append({
                **key_record,
                "table": table_name,
                "n_tied_rows": len(top_rows),
                "fields_present": max_fields,
                "source_rows": " | ".join(
                    top_rows["_source_row"]
                    .astype(str)
                    .tolist()
                ),
            })
        conflicting_fields = []
        for column in metadata_columns:
            values = (
                group[column]
                .dropna()
                .drop_duplicates()
            )
            if len(values) == 1:
                selected[column] = values.iloc[0]
            elif len(values) > 1:
                conflicting_fields.append(column)
                selected_top_values = (
                    top_rows[column]
                    .dropna()
                )
                if len(selected_top_values):
                    selected[column] = (
                        selected_top_values.iloc[0]
                    )
                elif pd.isna(selected[column]):
                    selected[column] = values.iloc[0]
                conflict_records.append({
                    **key_record,
                    "table": table_name,
                    "field": column,
                    "n_unique_values": len(values),
                    "observed_values": " | ".join(
                        repr(value)
                        for value in values.tolist()
                    ),
                    "selected_value": repr(
                        selected[column]
                    ),
                    "n_source_records": len(group),
                })
        selected["_source_record_count"] = len(group)
        selected["_selection_tie"] = len(top_rows) > 1
        selected["_n_conflicting_fields"] = len(
            conflicting_fields
        )
        selected["_conflicting_fields"] = (
            " | ".join(conflicting_fields)
            if conflicting_fields
            else pd.NA
        )
        selected_records.append(selected)
    collapsed = (
        pd.DataFrame(selected_records)
        .drop(
            columns=[
                "_fields_present",
                "_source_row",
            ]
        )
        .reset_index(drop=True)
    )
    conflicts = pd.DataFrame(conflict_records)
    ties = pd.DataFrame(tie_records)
    if collapsed.duplicated(list(keys)).any():
        raise AssertionError(
            f"{table_name} cleaning did not produce "
            "one row per key."
        )
    return collapsed, conflicts, ties

### Merge region sample data with SUBJECT metadata

In [ ]:
expected_sample_rows = len(SAMPLE)
subject_columns = [
    "ASAP_dataset_id",
    "ASAP_subject_id",
    "subject_id",
    "source_subject_id",
    "sex",
    "race",
    "ethnicity",
    "biobank_name",
    "gp2_phenotype",
]
subject_keys = [
    "ASAP_subject_id",
    "ASAP_dataset_id",
    "subject_id",
]
subject_clean, subject_conflicts, subject_selection_ties = collapse_duplicate_records(
    SUBJECT[subject_columns],
    subject_keys,
    table_name="SUBJECT"
)
print("Original SUBJECT rows:", len(SUBJECT))
print("Clean SUBJECT rows:", len(subject_clean))
print("SUBJECT conflicts:", len(subject_conflicts))
print("SUBJECT top-record ties:", len(subject_selection_ties))
df = pd.merge(
    SAMPLE,
    subject_clean,
    on=subject_keys,
    how="left",
    validate="many_to_one",
    indicator="_subject_merge",
)
assert len(df) == expected_sample_rows, "SUBJECT merge changed the SAMPLE row count."

### Merge region sample data with CLINPATH metadata

In [ ]:
clin_key = ["ASAP_dataset_id", "ASAP_subject_id"]
clin_grp = CLINPATH.groupby(clin_key)
# Rows-per-subject: a uniform "4" points to a structural merge artifact, not
# per-record error. The odd count out is the one to inspect by hand.
print("Rows per within-dataset subject:")
print(clin_grp.size().value_counts().sort_index(), "\n")
# Is there a column that is systematically distinct across the duplicate rows?
# If yes -> that's a real dimension (visit/instrument) and collapsing loses it.
# If no  -> the rows are fragments differing only by which fields are filled.
clin_dupe = clin_grp.filter(lambda g: len(g) > 1)
clin_value_cols = [c for c in CLINPATH.columns if c not in clin_key + ["ASAP_team_id",
              "subject_id", "source_subject_id"]]
per_group_max_distinct = (clin_dupe.groupby(clin_key)[clin_value_cols].nunique(dropna=True).max()
                              .sort_values(ascending=False))
print("Max distinct values any duplicate group shows, per field (top 8):")
print(per_group_max_distinct.head(8))
print("\n-> No column distinct in *every* row = fragmentation, safe to collapse.")
# Surface the odd-count subject explicitly (don't let .iloc[0] skip it)
odd = clin_grp.filter(lambda g: len(g) == 2)
if len(odd):
    print("\nOdd-count (2-row) subject — inspect directly:")
    display(odd.T)

In [ ]:
# Apply the agreed vocabulary maps FIRST so synonym-only differences
# (e.g. 'No PD nor other neurological disorder' == 'Healthy Control') do not
# masquerade as conflicts. Fill VOCAB_MAPS from the value_counts scan.
VOCAB_MAPS = {
    # (Deliberately empty / no primary_diagnosis mapping applied — see the
    #  "candidate harmonizations" note below. Add entries only once confirmed.)
}
CLINPATH_h = CLINPATH.copy()
for col, mapping in VOCAB_MAPS.items():
    if col in CLINPATH_h.columns:
        CLINPATH_h[col] = CLINPATH_h[col].replace(mapping)
# Collapse to one row per PERSON. This absorbs BOTH within-dataset fragments and
# cross-dataset resubmissions in a single pass; complementary fields fill,
# genuine value-vs-value disagreements are recorded in CLINPATH_conflicts.
CLINPATH_clean, CLINPATH_conflicts, CLINPATH_ties = collapse_duplicate_records(
    CLINPATH_h, ["ASAP_subject_id"], table_name="CLINPATH")
clin_multi         = CLINPATH_clean[CLINPATH_clean["_source_record_count"] > 1]
complementary = clin_multi[clin_multi["_n_conflicting_fields"] == 0]
conflicted    = clin_multi[clin_multi["_n_conflicting_fields"] > 0]
print(f"Person-level rows:              {len(CLINPATH_clean)}")
print(f"Subjects collapsed from >1 row: {len(clin_multi)}")
print(f"  ￿￿complementary (lossless):  {len(complementary)}   [Issue 1]")
print(f"  ￿￿TRUE disagreement:         {len(conflicted)}   [Issue 2 → review]")

In [ ]:
# One subject whose 4 rows each held different fields, collapsed to one complete row.
ex_id = complementary["ASAP_subject_id"].iloc[0]
raw = CLINPATH_h[CLINPATH_h["ASAP_subject_id"] == ex_id]
# show only the fields that were scattered (populated in some rows, null in others)
frag_cols = [c for c in clin_value_cols
             if raw[c].notna().any() and raw[c].isna().any()]
print(f"BEFORE — {ex_id}: {len(raw)} fragmented rows (note the scattered fills)")
display(raw[["ASAP_dataset_id"] + frag_cols].T)
print("AFTER — one reconstructed record (values filled from across the rows):")
display(CLINPATH_clean.loc[CLINPATH_clean["ASAP_subject_id"] == ex_id, frag_cols].T)

In [ ]:
before_merge = len(df)
# carry only the clinical payload; drop dataset/team/local-id + audit columns so
# person-level facts don't overwrite each sample's own dataset context
payload = [c for c in CLINPATH_clean.columns
           if c not in {"ASAP_dataset_id", "ASAP_team_id",
                        "subject_id", "source_subject_id"}
           and not c.startswith("_")]
df = df.merge(CLINPATH_clean[payload], on="ASAP_subject_id",
              how="left", validate="many_to_one", indicator="_clinpath_merge")
assert len(df) == before_merge, "CLINPATH merge changed the row count."
print("CLINPATH merge coverage:")
print(df["_clinpath_merge"].value_counts())
df = df.drop(columns="_clinpath_merge")

In [ ]:
# Uniform missing representation before saving/harmonizing. SAMPLE columns
# were merged raw, so SAMPLE-origin fields may still hold "NA"/"" while
# SUBJECT-/CLINPATH-origin fields are already pd.NA. Standardize once here.
df = standardize_missing(df)
print("Standardized missing values across", df.shape[1], "columns.")
df.head()

In [ ]:
df = df.dropna(axis=1, how='all')

In [ ]:
cleaned_metadata_path = OUTPUT_DIR / "asapcrn_cohort-pmdbs-sc-rnaseq_cleanedmetadata.csv"
df.to_csv(cleaned_metadata_path)